In [2]:
import os
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [3]:
# Load the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# Preprocess the data
# Drop irrelevant columns

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
# Encode categorical variables
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

In [6]:
# One Hot Encoding for Geography
data['Geography'].unique()

from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)

geo_encoder = onehot_encoder.fit_transform(data[['Geography']])

In [7]:
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
geo_encoded = pd.DataFrame(geo_encoder, columns = onehot_encoder.get_feature_names_out(['Geography']))

In [9]:
geo_encoded

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [10]:
# Combine one hot encoded columns with the original dataframe
data = pd.concat([data.drop('Geography', axis = 1), geo_encoded], axis = 1)

In [11]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [13]:
# Save the encoders and scalers
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

# Save the onehot encoder
with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)


In [12]:
# Divide the dataset into independent and dependent features
X = data.drop('EstimatedSalary', axis = 1)
y = data['EstimatedSalary']

In [14]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [15]:
# Scale these features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [16]:
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ANN Regression Problem Statement

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# Build our ANN Model
model = Sequential([
    Dense(64, activation='relu', input_shape = (x_train.shape[1],)), # First hidden layer which is connected with input layer
    Dense(32, activation='relu'),
    Dense(1) #Output
])

# Compile the model
model.compile(optimizer = 'adam', loss = 'mean_absolute_error', metrics = ['mae'])

model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [18]:
# Setup the tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "regression/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir = log_dir, histogram_freq = 1)

In [19]:
# Setup Early Stopping
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 10, restore_best_weights = True)

In [20]:
# Train the model
history = model.fit(
    x_train, y_train, validation_data = (x_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 2s 4ms/step - loss: 100369.3125 - mae: 100369.3125 - val_loss: 98488.6562 - val_mae: 98488.6562
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99541.2891 - mae: 99541.2891 - val_loss: 96824.9922 - val_mae: 96824.9922
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 96665.9688 - mae: 96665.9688 - val_loss: 92629.8516 - val_mae: 92629.8516
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 91040.0156 - mae: 91040.0156 - val_loss: 85674.5469 - val_mae: 85674.5469
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 82969.6719 - mae: 82969.6719 - val_loss: 76975.3047 - val_mae: 76975.3047
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 73785.5625 - mae: 73785.5625 - val_loss: 68100.6406 - val_mae: 68100.6406
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 65021.9648 - mae: 65021.964

In [21]:
# Load Tensorborad Extension
%load_ext tensorboard

In [23]:
%tensorboard --logdir regression/fit20250501-012305

Reusing TensorBoard on port 6006 (pid 18912), started 0:00:08 ago. (Use '!kill 18912' to kill it.)

In [24]:
# Evaluate model on the test data
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 2ms/step - loss: 50211.8828 - mae: 50211.8828
Test MAE : 50211.8828125


In [25]:
model.save('regression_model.h5')

d:\Udemy\Gen AI\Projects\ANN Classification\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
